<div dir="rtl" align="right">

# اختيارُ القنواتِ بِأهميّةِ السماتِ (Random Forest)

**مجموعةُ البياناتِ**: MOABB BNCI2014-001 (تخيّلٌ حركيّ)  
**القنواتُ**: 22 قناةً  
**معدّلُ أخذِ العيناتِ**: 250 Hz  
**المُشاركُ**: 1

---

## نظرةٌ عامّةٌ

نُدرّبُ غابةً عشوائيّةً ونَستخرجُ أهميّةَ كلِّ سمةٍ لِاختيارِ القنوات.

## المُخرجاتُ المُتوقّعةُ

- مخططٌ شريطيٌّ لِأهميّةِ السماتِ مُرتّبةً تنازليّاً
- خريطةُ الرأسِ بِحجمِ النقاطِ يُعبّرُ عن الأهميّة

## المُعاملاتُ الأساسيةُ

| المُعاملُ | القيمةُ |
| --- | --- |
| n_estimators | 100 |
| N_SELECT | 10 |

</div>

<div dir="rtl" align="right">

## 1. تثبيتُ المكتباتِ

</div>

In [ ]:
!pip install moabb mne scipy numpy plotly scikit-learn


<div dir="rtl" align="right">

## 2. تحميلُ مجموعةِ بياناتِ MOABB

تُنزّلُ MOABB البياناتِ تلقائيّاً عندَ أوّلِ استدعاءٍ (حوالي 44 ميجابايت).

</div>

In [ ]:
from moabb.datasets import BNCI2014_001
from moabb.paradigms import MotorImagery
import numpy as np

dataset = BNCI2014_001()
paradigm = MotorImagery(n_classes=2)
X, labels, meta = paradigm.get_data(dataset=dataset, subjects=[1])

mask = (labels == 'left_hand') | (labels == 'right_hand')
X = X[mask]
labels = labels[mask]

print(f'X shape: {X.shape}')
print(f'Labels: {np.unique(labels)}')
print(f'Trials: {len(labels)}')


<div dir="rtl" align="right">

## 3. استكشافُ البياناتِ

</div>

In [ ]:
n_trials, n_channels, n_samples = X.shape
print(f'Trials: {n_trials}')
print(f'Channels: {n_channels}')
print(f'Samples per trial: {n_samples}')
print(f'Trial duration: {n_samples/250:.2f} s')


<div dir="rtl" align="right">

## 4. تدريبُ الغابةِ العشوائيّةِ واستخراجُ الأهميّةِ

</div>

In [ ]:
from scipy.signal import welch
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

FS = 250
N_SELECT = 10
BANDS = [(8, 13, 'alpha'), (13, 30, 'beta')]

features = np.zeros((n_trials, n_channels * len(BANDS)))
for trial in range(n_trials):
    for ch in range(n_channels):
        freqs, psd = welch(X[trial, ch, :], fs=FS, nperseg=256)
        for b_idx, (fmin, fmax, bname) in enumerate(BANDS):
            mask_f = (freqs >= fmin) & (freqs <= fmax)
            features[trial, ch * len(BANDS) + b_idx] = np.trapezoid(psd[mask_f], freqs[mask_f])

X_train, X_test, y_train, y_test = train_test_split(
    features, labels, test_size=0.2, random_state=42, stratify=labels
)
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)
print(f"Test accuracy: {clf.score(X_test, y_test):.4f}")
importances = clf.feature_importances_
print(f'Top 5 features: {np.argsort(importances)[::-1][:5]}')

<div dir="rtl" align="right">

## 5. رسمٌ تفاعليٌّ

**علامَ تُلاحظُ؟**

- الأهميّةُ مُوزّعةٌ على عدةِ قنواتٍ معَ تَركّزٍ في المنطقةِ المركزيّة
- حجمُ النقاطِ يُعبّرُ عن الأهميّة

</div>

In [ ]:
import plotly.graph_objects as go

sorted_feat = np.argsort(importances)[::-1]
colors = ['green' if i < N_SELECT else 'gray' for i in range(len(importances))]
fig = go.Figure(go.Bar(x=[str(i) for i in sorted_feat], y=importances[sorted_feat], marker_color=colors, name='Importance'))
fig.update_layout(height=500, title='Random Forest Feature Importance', xaxis_title='Feature (sorted)', yaxis_title='Importance')
fig.show()


<div dir="rtl" align="right">

## خلاصةٌ

- الغابةُ العشوائيّةُ تُعطي أهميّةً لِكلِّ سمةٍ في تدريبٍ واحد
- أسرعُ من RFE لكنّهُ أقلُّ دقّةً لِأنّهُ لا يَأخذُ التفاعلاتِ تكراريّاً
- الأهميّةُ تَعكسُ مساهمةَ السمةِ في تَقليلِ عدمِ اليقين

</div>